# Задание №4. Построение нейронной сети для определения стиля текста (классификация текстов на основе CNN)

Данное задание представляет собой пошаговое руководство по созданию модели классификации текстов по стилю (например, новости, научные статьи, художественная литература) с использованием сверточных нейронных сетей (CNN). Вы познакомитесь с основными понятиями обработки естественного языка (NLP), научитесь подготавливать размеченный корпус, строить и обучать модель, а также использовать её для предсказания стиля нового текста. Задание адаптировано для выполнения в **Google Colab** (Jupyter Notebook). Все части работы (подготовка данных, обучение, сохранение, загрузка из репозитория и инференс) выполняются в одном ноутбуке.

---

## 1. Теоретическое введение: ключевые понятия NLP для классификации стиля текста

### 1.1. Корпус (Corpus)
Корпус – это структурированная коллекция текстов, используемая для обучения моделей NLP. В задаче классификации стиля необходим **размеченный корпус** – набор текстов, каждому из которых приписана метка класса (стиля). Примеры стилей: новости, научная статья, художественная литература, техническая документация и т.д.

### 1.2. Токенизация (Tokenization)
Процесс разбиения текста на минимальные единицы – **токены** (слова, знаки препинания). Для токенизации используем `Tokenizer` из Keras, который преобразует текст в последовательность целочисленных индексов.

### 1.3. Эмбеддинги (Embeddings)
Векторные представления слов. Слой **Embedding** преобразует индексы токенов в плотные векторы фиксированной размерности. Эти векторы обучаются вместе с моделью и позволяют улавливать семантические и синтаксические сходства между словами.

### 1.4. Сверточные нейронные сети (CNN) для текста
Сверточные сети изначально разработаны для обработки изображений, но успешно применяются и к текстам. **Одномерная свертка (Conv1D)** скользит по последовательности эмбеддингов, извлекая локальные признаки (например, характерные сочетания слов). Затем слой **GlobalMaxPooling1D** агрегирует самые важные признаки, после чего несколько полносвязных слоёв выполняют классификацию.

### 1.5. Классификация текста
Задача отнесения текста к одной из заранее известных категорий. Выходной слой модели использует активацию **softmax**, который выдаёт вероятности принадлежности каждому классу. Функция потерь – **categorical_crossentropy** (для one-hot меток) или **sparse_categorical_crossentropy** (для целочисленных меток).

### 1.6. Аугментация данных и балансировка классов
Для улучшения обобщающей способности модели и борьбы с дисбалансом классов применяют:
- **Аугментацию текста**: случайное удаление или перестановка слов (в пределах разумного).
- **Балансировку**: добавление синтетических примеров для классов с малым количеством образцов (oversampling) или уменьшение числа примеров для частых классов (undersampling). В коде используется аугментация для увеличения числа примеров до заданного минимума.

### 1.7. Метрики качества
Основная метрика – **accuracy** (доля правильных ответов). Для более детального анализа используют precision, recall, F1-score (библиотека `sklearn.metrics`).

---

## 2. Постановка задачи

Вам необходимо создать собственный репозиторий на GitHub, загрузить в него размеченный корпус текстов (файл `dataset.txt`), затем построить и обучить модель CNN для классификации стиля текста. После обучения вы сохраните модель, токенизатор и кодировщик меток, загрузите их в тот же репозиторий. В финальной части ноутбука вы продемонстрируете, как загрузить эти файлы из репозитория и использовать их для предсказания стиля новых текстов без повторного обучения. Весь код и отчёт должны быть оформлены в одном Jupyter Notebook.

Формат файла `dataset.txt`: каждая строка содержит текст и метку класса, разделённые символом табуляции (`\t`). Пример:
```
Это новость про политику.	news
В данной статье рассматривается квантовая физика.	science
На опушке леса стояла избушка.	fiction
```

---

## 3. Структура отчёта

Ваш отчёт должен содержать следующие разделы (в виде ячеек Markdown в ноутбуке):

1. **Титульный лист** (название работы, ФИО, группа, ссылка на репозиторий)
2. **Введение** (цель работы, краткое описание задачи)
3. **Теоретическая часть** (объяснение ключевых понятий: корпус, токенизация, эмбеддинги, CNN, классификация, аугментация, балансировка)
4. **Описание данных** (источник данных, статистика: количество примеров, распределение по классам, средняя длина текста; ссылка на файл в репозитории)
5. **Подготовка данных** (загрузка, аугментация, балансировка, токенизация, паддинг, разделение на train/test)
6. **Построение модели** (архитектура CNN, визуализация модели, summary)
7. **Обучение модели** (параметры обучения, графики потерь и точности)
8. **Инференс (предсказание)** (описание функции `predict_text`, демонстрация примеров)
9. **Сохранение модели и вспомогательных объектов** (код сохранения модели, токенизатора, label encoder; загрузка файлов в репозиторий)
10. **Загрузка модели из репозитория и быстрый инференс** (код загрузки через raw‑ссылку, повторное использование для предсказания)
11. **Выводы** (что получилось, какие были трудности, возможные улучшения)
12. **Список использованных источников**
13. **Приложение** (полный код с комментариями)

---

## 4. Требования к корпусу и ссылка на репозиторий

### 4.1. Минимальный и максимальный объём корпуса
- **Минимальный объём**: не менее 100 примеров на каждый класс (после балансировки). Если классов мало, общее количество должно быть достаточным для обучения (например, 500–1000 примеров).
- **Максимальный объём**: в Google Colab ограничение по оперативной памяти (обычно ~12–25 ГБ). Для комфортной работы рекомендуется использовать не более 50 000 примеров при длине текста до 200 слов. При превышении может потребоваться уменьшить размер словаря или использовать генераторы данных.

### 4.2. Создание репозитория и загрузка корпуса
1. Зарегистрируйтесь на [GitHub](https://github.com) (если у вас ещё нет аккаунта).
2. Создайте новый публичный репозиторий с названием, например, `text-style-classifier`.
3. Загрузите в репозиторий файл с корпусом `dataset.txt`. Вы можете использовать собственный размеченный корпус или сгенерировать синтетические данные с помощью кода ниже.  
   **Важно**: после загрузки получите **raw‑ссылку** на файл. Для этого откройте файл в репозитории, нажмите кнопку "Raw" и скопируйте URL. Он будет выглядеть примерно так:  
   `https://raw.githubusercontent.com/ваш_логин/text-style-classifier/main/dataset.txt`

### 4.3. Синтетический датасет (если нет реального)
Если у вас нет собственного корпуса, вы можете сгенерировать синтетические данные прямо в ноутбуке и затем сохранить их в файл для загрузки в репозиторий. Пример генерации трёх стилей:

```python
import random

def generate_synthetic_dataset(num_samples_per_class=200):
    styles = {
        'news': [
            "Вчера в городе произошло важное событие.",
            "Президент встретился с делегацией.",
            "Экономика продолжает расти.",
            "Новый закон вступил в силу.",
            "Спортивная команда одержала победу."
        ],
        'science': [
            "Исследование показало, что...",
            "Учёные открыли новую частицу.",
            "В статье рассматривается метод машинного обучения.",
            "Эксперимент подтвердил гипотезу.",
            "Формула описывает зависимость."
        ],
        'fiction': [
            "Наступил тихий вечер, и герой задумался.",
            "В лесу стояла странная тишина.",
            "Она открыла дверь и увидела свет.",
            "Ветер шептал таинственные истории.",
            "Замок возвышался над облаками."
        ]
    }
    
    texts, labels = [], []
    for style, phrases in styles.items():
        for i in range(num_samples_per_class):
            text = random.choice(phrases)
            # Добавим немного вариативности
            if random.random() > 0.5:
                text += " " + random.choice(phrases).split()[0] + "..."
            texts.append(text)
            labels.append(style)
    # Перемешаем
    combined = list(zip(texts, labels))
    random.shuffle(combined)
    texts, labels = zip(*combined)
    return list(texts), list(labels)

texts, labels = generate_synthetic_dataset(200)
with open('dataset.txt', 'w', encoding='utf-8') as f:
    for t, l in zip(texts, labels):
        f.write(f"{t}\t{l}\n")
```

Затем загрузите файл `dataset.txt` в свой репозиторий.

---

## 5. Пошаговое выполнение задания в Google Colab

### 5.1. Подготовка окружения

Импортируем необходимые библиотеки. В Google Colab они уже предустановлены.

```python
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from collections import Counter
import matplotlib.pyplot as plt
import pickle
import requests
import os
```

### 5.2. Загрузка корпуса из репозитория

Используйте вашу raw‑ссылку на файл `dataset.txt`.

```python
# Вставьте вашу ссылку
url_dataset = "https://raw.githubusercontent.com/ваш_логин/text-style-classifier/main/dataset.txt"

# Скачиваем файл
response = requests.get(url_dataset)
with open('dataset.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)

# Функция загрузки датасета
def load_dataset(path, num_samples=None):
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if num_samples and i >= num_samples:
                break
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            text, label = parts
            texts.append(text)
            labels.append(label)
    return texts, labels

texts, labels = load_dataset('dataset.txt')
print(f"Загружено примеров: {len(texts)}")
print(f"Распределение классов: {Counter(labels)}")
```

### 5.3. Аугментация и балансировка классов

Для улучшения модели и борьбы с дисбалансом используем функции аугментации и балансировки.

```python
# Простая аугментация: случайное удаление и перестановка слов
def augment_text(text):
    words = text.split()
    if len(words) <= 3:
        return text
    if random.random() < 0.5:
        idx = random.randrange(len(words))
        words.pop(idx)
    if random.random() < 0.5:
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return ' '.join(words)

# Балансировка классов через аугментацию
def balance_dataset(texts, labels, min_count=100):
    counter = Counter(labels)
    new_texts, new_labels = list(texts), list(labels)
    for label, count in counter.items():
        to_augment = [t for t, l in zip(texts, labels) if l == label]
        needed = max(0, min_count - count)
        for _ in range(needed):
            orig = random.choice(to_augment)
            new_t = augment_text(orig)
            new_texts.append(new_t)
            new_labels.append(label)
    return new_texts, new_labels

# Применяем балансировку (укажите желаемый минимум, например 100)
texts_bal, labels_bal = balance_dataset(texts, labels, min_count=100)
print(f"После балансировки: {len(texts_bal)} примеров")
print(f"Распределение: {Counter(labels_bal)}")
```

### 5.4. Токенизация и подготовка последовательностей

Зададим параметры токенизации и паддинга.

```python
VOCAB_SIZE = 10000          # максимальный размер словаря
OOV_TOKEN = '<OOV>'         # токен для неизвестных слов
MAX_LENGTH = 100            # максимальная длина текста (в словах)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(texts_bal)
sequences = tokenizer.texts_to_sequences(texts_bal)
X = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

# Кодирование меток
label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(labels_bal)
y = to_categorical(y_int)
num_classes = y.shape[1]
print(f"Число классов: {num_classes}")
print(f"Метки: {label_encoder.classes_}")
```

### 5.5. Разделение на обучающую и тестовую выборки

Используем стратифицированное разделение, чтобы сохранить пропорции классов.

```python
split_kwargs = {'test_size': 0.2, 'random_state': 42}
if min(Counter(y_int).values()) >= 2:
    split_kwargs['stratify'] = y_int
X_train, X_test, y_train, y_test = train_test_split(X, y, **split_kwargs)

print(f"Обучающих примеров: {len(X_train)}")
print(f"Тестовых примеров: {len(X_test)}")
```

### 5.6. Построение модели CNN

Используем архитектуру: Embedding → Conv1D → GlobalMaxPooling1D → Dense → Dropout → Dense.

```python
EMBEDDING_DIM = 100

model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()
```

### 5.7. Обучение модели

Обучим модель с EarlyStopping для предотвращения переобучения.

```python
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)
```

### 5.8. Визуализация процесса обучения

Построим графики потерь и точности.

```python
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери')

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Точность')
plt.show()
```

### 5.9. Функция предсказания стиля текста

Реализуем функцию, которая принимает строку и возвращает предсказанный класс.

```python
def predict_text(text: str):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    pred = model.predict(pad, verbose=0)
    return label_encoder.inverse_transform([np.argmax(pred)])[0]

# Примеры
test_phrases = [
    "Президент подписал новый указ.",
    "В статье описывается метод глубокого обучения.",
    "Наступила тихая ночь, и луна взошла над лесом."
]
for phrase in test_phrases:
    print(f"Текст: {phrase}\nПредсказанный стиль: {predict_text(phrase)}\n")
```

### 5.10. Сохранение модели и вспомогательных объектов

Сохраним модель, токенизатор и LabelEncoder. Эти файлы затем нужно загрузить в репозиторий.

```python
# Сохраняем модель
model.save('style_classifier.h5')
print("Модель сохранена в style_classifier.h5")

# Сохраняем токенизатор
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Токенизатор сохранён в tokenizer.pickle")

# Сохраняем LabelEncoder
with open('label_encoder.pickle', 'wb') as f:
    pickle.dump(label_encoder, f)
print("LabelEncoder сохранён в label_encoder.pickle")
```

### 5.11. Загрузка файлов в репозиторий

1. Скачайте файлы `style_classifier.h5`, `tokenizer.pickle` и `label_encoder.pickle` на свой компьютер (через панель файлов Colab).
2. В своём репозитории на GitHub нажмите "Add file" → "Upload files", загрузите эти три файла.
3. После загрузки получите raw‑ссылки на каждый файл. Для `.h5` лучше использовать ссылку вида `https://github.com/ваш_логин/text-style-classifier/raw/main/style_classifier.h5`, для pickle – аналогично.

### 5.12. Загрузка модели из репозитория и быстрый инференс

Этот блок можно выполнить после того, как вы загрузили файлы в репозиторий. Он демонстрирует, как загрузить сохранённые артефакты и использовать их для предсказания без повторного обучения.

```python
# Вставьте ваши ссылки
url_model = "https://github.com/ваш_логин/text-style-classifier/raw/main/style_classifier.h5"
url_tokenizer = "https://github.com/ваш_логин/text-style-classifier/raw/main/tokenizer.pickle"
url_encoder = "https://github.com/ваш_логин/text-style-classifier/raw/main/label_encoder.pickle"

# Скачиваем
!wget -O style_classifier.h5 {url_model}
!wget -O tokenizer.pickle {url_tokenizer}
!wget -O label_encoder.pickle {url_encoder}

# Загружаем
loaded_model = load_model('style_classifier.h5')
with open('tokenizer.pickle', 'rb') as f:
    loaded_tokenizer = pickle.load(f)
with open('label_encoder.pickle', 'rb') as f:
    loaded_encoder = pickle.load(f)

# Функция предсказания с загруженными объектами
def predict_from_loaded(text):
    seq = loaded_tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    pred = loaded_model.predict(pad, verbose=0)
    return loaded_encoder.inverse_transform([np.argmax(pred)])[0]

# Примеры
for phrase in test_phrases:
    print(f"Текст: {phrase}\nСтиль (загруженная модель): {predict_from_loaded(phrase)}\n")
```

### 5.13. Интерактивный режим (по желанию)

Вы можете добавить цикл для ввода текста и получения предсказания.

```python
print("=== Классификатор стиля текста (для выхода введите пустую строку) ===")
while True:
    txt = input("Введите текст: ").strip()
    if not txt:
        print("Выход.")
        break
    print("Стиль текста:", predict_from_loaded(txt))
```

---

## 6. Полный код нейронной сети с обучением

Для удобства ниже приведён сводный код всех ячеек, необходимых для полного цикла обучения модели (без интерактива). Вы можете скопировать его в одну ячейку или выполнять по частям.

```python
# Импорты
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from collections import Counter
import matplotlib.pyplot as plt
import pickle
import requests

# Загрузка датасета из репозитория
url_dataset = "https://raw.githubusercontent.com/ваш_логин/text-style-classifier/main/dataset.txt"
response = requests.get(url_dataset)
with open('dataset.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)

def load_dataset(path):
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            texts.append(parts[0])
            labels.append(parts[1])
    return texts, labels

texts, labels = load_dataset('dataset.txt')
print(f"Загружено примеров: {len(texts)}")
print(f"Распределение: {Counter(labels)}")

# Аугментация и балансировка
def augment_text(text):
    words = text.split()
    if len(words) <= 3:
        return text
    if random.random() < 0.5:
        idx = random.randrange(len(words))
        words.pop(idx)
    if random.random() < 0.5:
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return ' '.join(words)

def balance_dataset(texts, labels, min_count=100):
    counter = Counter(labels)
    new_texts, new_labels = list(texts), list(labels)
    for label, count in counter.items():
        to_augment = [t for t, l in zip(texts, labels) if l == label]
        needed = max(0, min_count - count)
        for _ in range(needed):
            orig = random.choice(to_augment)
            new_t = augment_text(orig)
            new_texts.append(new_t)
            new_labels.append(label)
    return new_texts, new_labels

texts, labels = balance_dataset(texts, labels, min_count=100)
print(f"После балансировки: {len(texts)} примеров")

# Токенизация
VOCAB_SIZE = 10000
OOV_TOKEN = '<OOV>'
MAX_LENGTH = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

# Кодирование меток
label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(labels)
y = to_categorical(y_int)
num_classes = y.shape[1]

# Разделение
split_kwargs = {'test_size': 0.2, 'random_state': 42}
if min(Counter(y_int).values()) >= 2:
    split_kwargs['stratify'] = y_int
X_train, X_test, y_train, y_test = train_test_split(X, y, **split_kwargs)

# Модель
model = Sequential([
    Embedding(VOCAB_SIZE, 100, input_length=MAX_LENGTH),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Обучение
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=50, batch_size=32,
                    validation_data=(X_test, y_test), callbacks=[early_stop], verbose=1)

# Графики
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Loss')
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Accuracy')
plt.show()

# Сохранение
model.save('style_classifier.h5')
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)
with open('label_encoder.pickle', 'wb') as f:
    pickle.dump(label_encoder, f)
print("Модель и вспомогательные файлы сохранены.")
```

---

## 7. Код с загруженной моделью для быстрого запуска

Этот блок можно использовать отдельно, если модель уже сохранена в репозитории. Он загружает файлы по raw‑ссылкам и выполняет предсказание.

```python
# Укажите ваши ссылки
url_model = "https://github.com/ваш_логин/text-style-classifier/raw/main/style_classifier.h5"
url_tokenizer = "https://github.com/ваш_логин/text-style-classifier/raw/main/tokenizer.pickle"
url_encoder = "https://github.com/ваш_логин/text-style-classifier/raw/main/label_encoder.pickle"

!wget -O style_classifier.h5 {url_model}
!wget -O tokenizer.pickle {url_tokenizer}
!wget -O label_encoder.pickle {url_encoder}

import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

loaded_model = load_model('style_classifier.h5')
with open('tokenizer.pickle', 'rb') as f:
    tokenizer = pickle.load(f)
with open('label_encoder.pickle', 'rb') as f:
    label_encoder = pickle.load(f)

MAX_LENGTH = 100  # должно совпадать с использованным при обучении

def predict_style(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    pred = loaded_model.predict(pad, verbose=0)
    return label_encoder.inverse_transform([np.argmax(pred)])[0]

# Пример использования
print(predict_style("Президент подписал новый указ."))
```

---

## 8. Задания для студентов

### Задание 1. Подготовка репозитория и данных
- Создайте публичный репозиторий на GitHub.
- Подготовьте размеченный корпус текстов в формате `текст\tметка` (или сгенерируйте синтетический) и загрузите его в репозиторий под именем `dataset.txt`. Убедитесь, что объём корпуса соответствует требованиям (минимум 100 примеров на класс после балансировки).
- Получите raw‑ссылку на файл и запишите её для использования в ноутбуке.

### Задание 2. Подготовка данных в ноутбуке
- В ноутбуке напишите код для загрузки корпуса из репозитория по ссылке.
- Выведите статистику: количество примеров, распределение по классам, среднюю длину текста.
- Примените аугментацию и балансировку, чтобы каждый класс имел не менее 100 примеров. Объясните, зачем это нужно.
- Выполните токенизацию, паддинг и разделите данные на обучающую и тестовую выборки (80/20) со стратификацией.

### Задание 3. Построение модели
- Постройте модель CNN с архитектурой, аналогичной примеру (Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout). Можно экспериментировать с количеством фильтров, размером ядра и размером эмбеддингов.
- Выведите summary модели.
- Скомпилируйте модель с оптимизатором 'adam' и функцией потерь 'categorical_crossentropy'.

### Задание 4. Обучение модели
- Обучите модель с использованием `EarlyStopping` (patience=5) на 50 эпохах.
- Постройте графики потерь и точности на обучающей и тестовой выборках.
- Проанализируйте, не переобучилась ли модель.

### Задание 5. Предсказание стиля текста (с текущей моделью)
- Реализуйте функцию `predict_text`, которая принимает строку и возвращает предсказанный класс.
- Протестируйте её на 3–5 примерах (можно взять из тестовой выборки или придумать свои). Выведите текст и предсказанный стиль.

### Задание 6. Сохранение и загрузка в репозиторий
- Сохраните обученную модель в файл `style_classifier.h5`, токенизатор – в `tokenizer.pickle`, label encoder – в `label_encoder.pickle`.
- Загрузите эти три файла в свой репозиторий на GitHub.
- Получите raw‑ссылки на каждый файл (используйте формат `https://github.com/.../raw/main/...`).

### Задание 7. Загрузка модели из репозитория и быстрый инференс
- В отдельной ячейке напишите код, который скачивает файлы модели, токенизатора и энкодера из репозитория по ссылкам.
- Загрузите модель и вспомогательные объекты, и с их помощью предскажите стиль для тех же примеров, что и в задании 5. Убедитесь, что результаты совпадают (с учётом возможных погрешностей округления).

### Задание 8. Анализ результатов
- Оцените качество модели на тестовой выборке, рассчитайте accuracy. Если возможно, постройте матрицу ошибок (confusion matrix) и выведите classification report (precision, recall, F1).
- В каких классах модель ошибается чаще всего? Почему?
- Предложите способы улучшения модели (например, увеличение корпуса, использование предобученных эмбеддингов, изменение архитектуры, добавление регуляризации).

### Задание 9*. Дополнительно (по желанию)
- Попробуйте использовать двунаправленный LSTM вместо CNN или комбинацию CNN + LSTM. Сравните результаты.
- Реализуйте загрузку предобученных эмбеддингов (например, FastText или Word2Vec) и используйте их в модели.
- Добавьте в pipeline очистку текста (удаление стоп-слов, пунктуации, лемматизацию) и посмотрите, влияет ли это на качество.

---

## 9. Заключение

В ходе выполнения этого задания вы:
- создали собственный репозиторий на GitHub и научились загружать туда файлы;
- познакомились с основными этапами создания модели классификации текстов на основе CNN;
- подготовили размеченные данные, применили аугментацию и балансировку;
- построили и обучили свёрточную нейронную сеть;
- освоили сохранение модели, токенизатора и кодировщика меток, а затем их загрузку из репозитория для повторного использования;
- научились анализировать качество модели и предлагать пути улучшения.

Полученные навыки являются фундаментом для решения широкого круга задач NLP, включая анализ тональности, тематическое моделирование, определение автора и многие другие.